# Task A -- does task-adaptive pretraining help?

TAPT is the largest measured effect anywhere in this project: on Task B it was worth
**+2.9 points** over stock MuRIL on identical folds, and it was the only intervention of
roughly a dozen that beat the reference. Task A has never tested it.

This notebook runs a matched pair five-fold, so the answer comes from 6,401 out-of-fold
rows where noise is about 0.6 points.

| setting | value |
|---|---|
| TAPT corpus | Task A comments + OffensEval Kannada, text only, no labels read |
| TAPT schedule | 8 MLM epochs, `--val-frac 0`, `--min-words 1` |
| classifier | demojized MuRIL, `--reinit-layers 1`, 6 epochs, effective batch 16 |
| evaluation | five-fold OOF, split seed 42, `--select last` |
| the one difference | stock `google/muril-base-cased` vs the TAPT checkpoint |

## An honest caveat about this design

The TAPT stage reads every Task A training comment, including the rows each fold later
scores itself on. No labels are read, because masked language modelling needs text alone,
but the TAPT arm has still seen the wording of its own out-of-fold rows and the control
has not. **The comparison is therefore biased in TAPT's favour.**

This is the same convention Task B used, which is what makes the numbers comparable to
that +2.9. The leak-free alternative already exists as
`01_tapt_demojized_muril.ipynb`, which builds TAPT from the 85% training side only and
scores on a fixed 960-row holdout. That design is unbiased but its noise is about 1.3
points, so it cannot resolve a one-point effect.

Read the two together. If this notebook shows a large gain and the holdout notebook agrees
in direction, TAPT is real. If this one shows under a point, the bias alone could explain
it and TAPT should not be adopted for Task A.

## Runtime

About **6.3 hours**: 60 minutes for TAPT, then 160 per five-fold arm. The guard runs the
TAPT arm first, so a short session still produces the number that matters most.

Set **Accelerator** to `GPU T4 x2` or `GPU P100` and **Internet** on, then
**Save Version -> Save & Run All**.

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("repo:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import numpy as np
import pandas as pd
import torch
from hastika.common.preprocessing import dedupe_index

assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))
train = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(train["Comment"].tolist(), train["Label"].tolist(), "task A")
print(f"raw labelled rows: {len(train)}; deduplicated rows used for fitting: {len(keep)}")
assert len(keep) == 6401, len(keep)

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Build the TAPT checkpoint

`hastika.task_b.tapt` is the shared MLM implementation. The module lives under `task_b`
for historical reasons but reads whatever corpus it is given. Here it is pointed at Task
A's own comments plus the permitted external Kannada text. Task B's files are excluded and
no label column is read from either input.

`--allow-transductive` is required because the loader gates Task A's file behind it, a
guard written for Task B where those comments are its test set. For Task A they are its
own training text, so the flag is simply unlocking the right corpus.

In [ ]:
import time
t0 = time.time()
BUDGET_H, RESERVE_MIN = 10.5, 20
left = lambda: BUDGET_H * 3600 - (time.time() - t0) - RESERVE_MIN * 60

from sklearn.metrics import f1_score
from hastika.common.preprocessing import clean
df = train.iloc[keep].reset_index(drop=True)
X = np.array([clean(t, demojize=True) for t in df["Comment"]])
y = (df["Label"] == "Hate").astype(int).values
oof_of = lambda tag: np.load(pathlib.Path("artifacts/runs") / tag / "oof_probs.npy")
score_of = lambda tag: f1_score(y, oof_of(tag).argmax(1), average="macro")

TAPT_OUT = "artifacts/runs/tapt-task-a"
if (pathlib.Path(TAPT_OUT) / "config.json").exists():
    print("using existing TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--corpus", "data/raw/binary_train.csv", "data/external/offenseval_kn.csv",
         "--val-frac", "0", "--min-words", "1", "--no-dedupe", "--epochs", "8",
         "--allow-transductive", "--out", TAPT_OUT],
        log="artifacts/logs/task_a_tapt.log")

## 2. Five folds for each arm

Identical in every respect except the starting encoder. Both use one reinitialized layer.
If Run 11 finds two layers better for Task A, rerun with `--reinit-layers 2`; the
conclusion should not change, because the comparison here is between encoders, not between
reinitialization settings.

In [ ]:
COMMON = ["--folds", "5", "--epochs", "6", "--bs", "8", "--grad-accum", "2",
          "--eval-bs", "32", "--select", "last", "--reinit-layers", "1", "--seeds", "42"]
ARMS = [("task_a_tapt_5f", ["--model", TAPT_OUT]),
        ("task_a_stock_5f", ["--model", "google/muril-base-cased"])]
ran = []
for tag, extra in ARMS:
    if left() < 160 * 60:
        print(f"skip {tag}: {left()/60:.0f} min left, needs ~160", flush=True)
        continue
    run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", tag,
         *COMMON, *extra], log=f"artifacts/logs/{tag}.log")
    ran.append(tag)
print("\narms completed:", ran)

## 3. The comparison

Both figures are out-of-fold macro-F1 over all 6,401 rows: no classifier scored a row it
trained on. The bias described at the top lives in the TAPT stage, not here.

In [ ]:
for tag in ran:
    print(f"  {tag:20s} OOF macro-F1 {score_of(tag):.4f}")
if len(ran) == 2:
    delta = score_of("task_a_tapt_5f") - score_of("task_a_stock_5f")
    print(f"\nTAPT effect on Task A: {delta:+.4f}")
    print(f"  Task B measured {0.6013 - 0.5727:+.4f} under the same convention")
    print("  fold noise here is about 0.006; treat anything smaller as unresolved")

from sklearn.metrics import classification_report
for tag in ran:
    print(f"\n=== {tag} ===")
    print(classification_report(y, oof_of(tag).argmax(1),
                                target_names=["Non-Hate", "Hate"], digits=3))

## 4. Preserve outputs

Keep the TAPT checkpoint only if the gain justifies it: it is about a gigabyte. The
`oof_probs.npy` files are small and are what let any later blend or threshold experiment
run without a GPU.

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_tapt_outputs")
OUT.mkdir(parents=True, exist_ok=True)
for tag in ran:
    for name in ["oof_probs.npy", "test_probs.npy", "predictions.csv"]:
        p = pathlib.Path("artifacts/runs") / tag / name
        if p.exists():
            shutil.copy2(p, OUT / f"{tag}_{name}")
    shutil.copy2(f"artifacts/logs/{tag}.log", OUT / f"{tag}.log")
if pathlib.Path("artifacts/logs/task_a_tapt.log").exists():
    shutil.copy2("artifacts/logs/task_a_tapt.log", OUT / "task_a_tapt.log")
json.dump({t: score_of(t) for t in ran}, open(OUT / "oof_scores.json", "w"), indent=2)
print(sorted(x.name for x in OUT.iterdir()))

## 5. What to do with the result

Record both out-of-fold numbers in `docs/EXPERIMENTS.md`, including the losing arm.

If TAPT wins by more than about a point it becomes part of the Task A recipe, and the next
step is a full-data fit with it, blended at whatever weight Run 11 establishes. If it wins
by less, remember the bias at the top favours it, and check `01_tapt_demojized_muril.ipynb`
before adopting it.

This notebook deliberately produces no submission. Deciding and building are separate
steps, and a five-fold run is for deciding.